# Qwen3.5-0.8B serving smoke experiment

This notebook is the analysis and launch surface for `qwen35-0.8b-smoke`. It loads the real YAML profile, expands the same matrix as the CLI, reads the harness's canonical `manifest.json` and `summary.json` artifacts, and keeps quality evidence separate from performance.

**Safe default:** Run All is offline and read-only. It does not connect over SSH, provision a Pod, invoke the runner, or make a paid call. The launch cell runs only after `RUN_EXPERIMENT = True` is set deliberately.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Markdown, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'experiment_profiles' / 'qwen35-0.8b-smoke.yaml').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from within the llm-serving checkout.')

PROJECT_ROOT = find_project_root(Path.cwd())
PROFILE_PATH = PROJECT_ROOT / 'experiment_profiles' / 'qwen35-0.8b-smoke.yaml'
OUTPUT_ROOT = Path(os.environ.get('LLM_SERVING_OUTPUT_ROOT', PROJECT_ROOT / 'output')).expanduser().resolve()

from llm_serving.analysis import (
    comparable_quality_groups, discover_runs, history_rows, performance_rows,
    plot_baseline_deltas, plot_performance, plot_quality, quality_rows, write_run_notes,
)
from llm_serving.matrix import expand_matrix
from llm_serving.schemas import ExperimentProfile

print(f'Project: {PROJECT_ROOT}')
print(f'Artifact root: {OUTPUT_ROOT}')

## Question and hypothesis

The smoke profile asks whether CUDA graph execution improves Qwen3.5-0.8B serving performance for short-interactive, long-prefill, near-limit-prefill, and decode-heavy traffic while chunked prefill remains enabled for the model's aligned Mamba cache mode.

The working hypothesis is that CUDA graphs increase output throughput and reduce inter-token latency most clearly on decode-heavy traffic. LongBench v2, GSM8K, and SuperGLUE are separate smoke-quality checks; their 20-example results are not full benchmark scores and must not be used to explain a performance delta. The 256K serving cap is the model's full advertised context window; this 20-example LongBench v2 slice remains a smoke result, not a leaderboard result.

In [ ]:
profile_raw = yaml.safe_load(PROFILE_PATH.read_text(encoding='utf-8'))
profile = ExperimentProfile.model_validate(profile_raw)
display(Markdown(f'**Loaded profile:** `{PROFILE_PATH}`  \n**Model:** `{profile.model.id}` at revision `{profile.model.revision}`'))
pd.json_normalize(profile.model_dump(), sep='.').T.rename(columns={0: 'resolved value'})

## Resolved experiment matrix

This uses the harness's matrix expansion rather than reimplementing it in the notebook. The first row is the within-run baseline used by the saved reports.

In [ ]:
matrix = expand_matrix(profile)
matrix_df = pd.DataFrame([
    {
        'case_id': case.case_id, 'baseline': case.is_baseline,
        'cuda_graphs': case.cuda_graphs, 'chunked_prefill': case.chunked_prefill,
        'repetition': case.repetition, 'vllm_args': shlex.join(case.vllm_args),
    }
    for case in matrix
])
matrix_df

## Explicit launch controls

Edit the host alias and inventory path for an existing SSH host or a prepared temporary inventory. The existing harness command runs only when the switch in the next cell is changed to `True`.

Set the hypothesis before launch so the new run receives an `experiment-notes.json` sidecar. The notebook creates that sidecar only for the newly created run and refuses to replace an existing notes file.

In [ ]:
HOST = os.environ.get('LLM_SERVING_HOST', 'replace-with-host-alias')
HOST_INVENTORY = Path(os.environ.get('LLM_SERVING_HOST_INVENTORY', PROJECT_ROOT / 'hosts.local.yaml')).expanduser()
HYPOTHESIS = 'CUDA graphs improve decode-heavy throughput and ITL on this host.'
INITIAL_OBSERVATIONS = ''

plan_command = [sys.executable, '-m', 'llm_serving.cli', 'plan', str(PROFILE_PATH), '--host', HOST, '--inventory', str(HOST_INVENTORY)]
run_command = [sys.executable, '-m', 'llm_serving.cli', 'run', str(PROFILE_PATH), '--host', HOST, '--inventory', str(HOST_INVENTORY), '--output-dir', str(OUTPUT_ROOT)]
display(Markdown('**Offline plan preview**\n```shell\n' + shlex.join(plan_command) + '\n```'))
display(Markdown('**Existing-harness run preview**\n```shell\n' + shlex.join(run_command) + '\n```'))

In [ ]:
RUN_EXPERIMENT = os.environ.get('LLM_SERVING_RUN_EXPERIMENT', '').lower() in {'1', 'true', 'yes'}  # Deliberate opt-in: may use SSH/GPU resources and incur cost.

if not RUN_EXPERIMENT:
    print('Analysis-only mode: no validation, SSH, provisioning, or experiment run was started.')
else:
    if HOST == 'replace-with-host-alias':
        raise ValueError('Set LLM_SERVING_HOST or edit HOST before launching.')
    if not HOST_INVENTORY.is_file():
        raise FileNotFoundError(f'Host inventory not found: {HOST_INVENTORY}')
    run_parent = OUTPUT_ROOT / profile.name
    before = {path for path in run_parent.iterdir() if path.is_dir() and (path / 'summary.json').is_file()} if run_parent.is_dir() else set()
    completed = subprocess.run(run_command, cwd=PROJECT_ROOT, check=False)
    after = {path for path in run_parent.iterdir() if path.is_dir() and (path / 'summary.json').is_file()}
    created = sorted(after - before)
    if len(created) == 1:
        notes_path = write_run_notes(created[0], hypothesis=HYPOTHESIS, observations=INITIAL_OBSERVATIONS)
        print(f'Run notes created: {notes_path}')
    else:
        raise RuntimeError(f'Expected one new run directory, found {created}')
    print(f'Harness exit code: {completed.returncode}. Failed runs remain available for analysis.')

## Saved run history

The reader includes failed and partial runs. Missing files and metrics remain visible warnings or blank values; they are never replaced with zero. If no compatible measured run exists, tables and charts state that results are unavailable.

In [ ]:
all_measured_runs = discover_runs(OUTPUT_ROOT, profile_name=profile.name, source='measured')

def recorded_context_limit(run):
    return ((run.get('profile') or {}).get('server') or {}).get('max_model_len')


measured_runs = [run for run in all_measured_runs if recorded_context_limit(run) == profile.server.max_model_len]
incompatible_runs = [run for run in all_measured_runs if recorded_context_limit(run) != profile.server.max_model_len]
history = measured_runs
if incompatible_runs:
    excluded = ', '.join(run['run_id'] for run in incompatible_runs)
    display(Markdown(f'> **Excluded measured run(s):** `{excluded}` used a different `server.max_model_len`; set `LLM_SERVING_RUN_IDS` only after running this 256K profile.'))

if measured_runs:
    display(Markdown(f'**Measured artifact history:** {len(measured_runs)} run(s) found.'))
else:
    display(Markdown('> **No compatible measured runs found.** Run this 256K profile to populate the analysis.'))
pd.DataFrame(history_rows(history))

## Performance results

These panels show throughput, TTFT p50/p95, ITL p50/p95, and end-to-end latency only where the artifacts record them. Blue bars are baseline cases and orange bars are alternatives. Variability is not drawn because the current profile has one repetition per case; increase repetitions before interpreting run-to-run spread.

In [ ]:
measured_performance = performance_rows(measured_runs)
requested_ids = {value.strip() for value in os.environ.get('LLM_SERVING_RUN_IDS', '').split(',') if value.strip()}
if measured_performance:
    if requested_ids:
        analysis_runs = [run for run in measured_runs if run['run_id'] in requested_ids]
        if not performance_rows(analysis_runs):
            raise ValueError(f'LLM_SERVING_RUN_IDS selected no runs with performance rows: {sorted(requested_ids)}')
    else:
        analysis_runs = [next((run for run in measured_runs if run['complete'] and performance_rows([run])), next(run for run in measured_runs if performance_rows([run])))]
    display(Markdown('**Data source: measured harness artifacts.** Charts show the latest complete run by default; set `LLM_SERVING_RUN_IDS` to a comma-separated list for an explicit comparison.'))
else:
    analysis_runs = []
    display(Markdown('> **No compatible measured performance results.** Charts below show unavailable panels until this 256K profile completes.'))
perf = performance_rows(analysis_runs)
perf_df = pd.DataFrame(perf)
columns = ['run_id', 'source', 'case_id', 'case_status', 'config', 'workload', 'output_throughput_tok_per_s', 'ttft_p50_ms', 'ttft_p95_ms', 'itl_p50_ms', 'itl_p95_ms', 'e2e_p50_ms']
perf_df.reindex(columns=columns)

In [ ]:
plot_performance(perf);


## Within-run baseline deltas

Deltas come directly from each canonical summary. Throughput gains are favorable; latency reductions are favorable. Cross-run deltas are not synthesized when hardware, runtime, or workload identity may differ.

In [ ]:
plot_baseline_deltas(perf);


## Quality evidence (separate from performance)

Quality rows are grouped only when their recorded task settings and named metric match. That check cannot establish compatibility for metadata the current artifacts do not record, such as evaluator version or prompt-template revision. LongBench v2, GSM8K, and SuperGLUE each use a 20-example smoke subset here; they are not leaderboard estimates. The profile uses the model's full 256K advertised context window.

In [ ]:
measured_quality = quality_rows(measured_runs)
if measured_quality:
    quality = measured_quality
    display(Markdown('**Data source: measured quality artifacts.**'))
    longbench_configured = any(task.name == 'longbench2' for task in profile.quality.tasks)
    longbench_reported = any(row['task_name'] == 'longbench2' for row in quality)
    if longbench_configured and not longbench_reported:
        display(Markdown('> **Incomplete quality suite:** LongBench v2 is configured but has no recorded result. Do not treat this run as a complete quality evaluation.'))
else:
    quality = []
    display(Markdown('> **No compatible measured quality results.**'))
quality_df = pd.DataFrame(quality)
display(quality_df)
groups = comparable_quality_groups(quality)
print(f'Groups sharing recorded task settings and metric: {len(groups)}')
plot_quality(quality);

## Observations and next decision

After a real run, record what happened in that run's `experiment-notes.json` sidecar and rerun the analysis cells. Check completion and warning columns before comparing values. Treat missing E2E or quality values as unavailable. With one repetition, use the result to choose a focused repeat rather than claim stable variability.

Suggested decision: if the measured decode-heavy delta agrees with the hypothesis, repeat both cases at least three times on the same host/runtime. If a case failed, inspect its saved server log and keep the partial run in history.